# 04 - Cleaning Eurostat First Permits for Cameroonian Citizens

This notebook cleans Eurostat first permit data for Cameroonian citizens in European destination countries.

Analytical role:

- supports Q2 on observable entry reasons
- separates family, education and employment-related permit categories
- converts wide Excel sheets into a consistent long format
- prepares a standardized file used later to build `entry_reasons_master.csv`

The notebook preserves the original Eurostat categories where they are needed for traceability.


In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
PROJECT_ROOT = Path("..")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

EUROPE_PATH = DATA_RAW / "europe"
CLEANED_PATH = DATA_PROCESSED / "cleaned"

CLEANED_PATH.mkdir(parents=True, exist_ok=True)

In [ ]:
xls = pd.ExcelFile(EUROPE_PATH / "eurostat_first_permits_cameroon.xlsx")
xls.sheet_names

In [ ]:
sheet1_raw = pd.read_excel(
    EUROPE_PATH / "eurostat_first_permits_cameroon.xlsx",
    sheet_name="Feuille 1",
    header=None
)

sheet1_raw

In [ ]:
sheet1_raw = sheet1_raw.drop(columns=[4, 6, 8, 10, 12, 14, 16, 18, 20])

In [ ]:
sheet1_raw = sheet1_raw.drop(columns=[2])
sheet1_raw

In [ ]:
sheet1_raw = sheet1_raw.drop(index=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 47, 48, 49, 50, 51, 52])
sheet1_raw

In [ ]:
sheet1_raw = sheet1_raw.replace(":", 0)
sheet1_raw = sheet1_raw[sheet1_raw[0] != "GEO (Libellés)"]
sheet1_raw = sheet1_raw.reset_index(drop=True)
sheet1_raw

In [ ]:
sheet1_raw = sheet1_raw.drop(index=[1, 2])
sheet1_raw

In [ ]:
sheet1_raw = sheet1_raw.rename(columns={0: "Date"})
sheet1_raw

In [ ]:
sheet1_raw.columns = sheet1_raw.iloc[0]
sheet1_raw = sheet1_raw.iloc[1:].reset_index(drop=True)

sheet1_raw = sheet1_raw.rename(columns={"TIME": "Date"})
sheet1_raw = sheet1_raw[sheet1_raw["Date"] != "GEO (Libellés)"]

sheet1_raw["motif"] = "famille"

sheet1_raw


In [ ]:
sheet1_raw = sheet1_raw.drop(columns=["motif"], errors="ignore")


In [ ]:
sheet1_raw

In [ ]:
def reshape_first_permits_long(df, motif):
    df = df.copy()

    # Add the entry reason
    df["motif"] = motif

    # Reshape from wide format to long format
    df_long = df.melt(
        id_vars=["destination_country", "motif"],
        var_name="year",
        value_name="permits"
    )

    # Clean data types
    df_long["year"] = df_long["year"].astype(int)
    df_long["permits"] = pd.to_numeric(df_long["permits"], errors="coerce").fillna(0).astype(int)

    return df_long


In [ ]:
sheet1_raw = sheet1_raw.rename(columns={"Date": "destination_country"})
sheet1_raw

In [ ]:
def reshape_first_permits_long(df, motif):
    df = df.copy()

    # Add the entry reason
    df["motif"] = motif

    # Reshape from wide format to long format
    df_long = df.melt(
        id_vars=["destination_country", "motif"],
        var_name="year",
        value_name="permits"
    )

    # Clean data types
    df_long["year"] = df_long["year"].astype(int)
    df_long["permits"] = pd.to_numeric(df_long["permits"], errors="coerce").fillna(0).astype(int)

    return df_long


In [ ]:
sheet1_raw = reshape_first_permits_long(sheet1_raw, "famille")
sheet1_raw

## Source Structure Check

Before cleaning the full file, the notebook inspects the relevant Excel sheets to identify:

- where the actual data starts
- which rows contain headers
- whether years are stored as columns
- where destination country names are located
- whether Eurostat flags or metadata rows need to be excluded


In [ ]:
sheet2_raw = pd.read_excel(
    EUROPE_PATH / "eurostat_first_permits_cameroon.xlsx",
    sheet_name="Feuille 2",
    header=None
)

sheet2_raw

In [ ]:
def clean_sheet2_etude(excel_path):
    sheet2_raw = pd.read_excel(
        excel_path,
        sheet_name="Feuille 2",
        header=None
    )

    # Remove flag columns 2, 4, 6, ..., 20
    sheet2_raw = sheet2_raw.drop(columns=list(range(2, 21, 2)))

    # Remove unused rows, keeping row 10 as the header
    rows_to_drop = list(range(0, 10)) + [11, 12, 13] + list(range(47, 53))
    sheet2_raw = sheet2_raw.drop(index=rows_to_drop)

    # Replace ":" with 0
    sheet2_raw = sheet2_raw.replace(":", 0)

    # Use row 10 as the header
    sheet2_raw.columns = sheet2_raw.iloc[0]
    sheet2_raw = sheet2_raw.iloc[1:].reset_index(drop=True)

    # Rename TIME to destination_country
    sheet2_raw = sheet2_raw.rename(columns={"TIME": "destination_country"})

    # Add the reason column
    sheet2_raw["motif"] = "etude"

    return sheet2_raw


In [ ]:
sheet2_raw = clean_sheet2_etude(
    EUROPE_PATH / "eurostat_first_permits_cameroon.xlsx"
)

sheet2_raw


In [ ]:
def reshape_first_permits_long(df):
    df_long = df.melt(
        id_vars=["destination_country", "motif"],
        var_name="year",
        value_name="permits"
    )

    df_long["year"] = df_long["year"].astype(int)
    df_long["permits"] = (
        pd.to_numeric(df_long["permits"], errors="coerce")
        .fillna(0)
        .astype(int)
    )

    df_long = df_long[
        ["destination_country", "motif", "year", "permits"]
    ]

    return df_long


In [ ]:
sheet2_raw = reshape_first_permits_long(sheet2_raw)
sheet2_raw


In [ ]:
def clean_and_reshape_first_permits(excel_path, sheet_name, motif):
    df = pd.read_excel(
        excel_path,
        sheet_name=sheet_name,
        header=None
    )

    # Remove flag columns 2, 4, 6, ..., 20
    df = df.drop(
        columns=list(range(2, 21, 2)),
        errors="ignore"
    )

    # Remove unused rows
    rows_to_drop = list(range(0, 10)) + [11, 12, 13] + list(range(47, 53))

    df = df.drop(
        index=rows_to_drop,
        errors="ignore"
    )

    # Replace ":" with 0
    df = df.replace(":", 0)

    # Use row 10 as the header
    df.columns = df.iloc[0]
    df = df.iloc[1:].reset_index(drop=True)

    # Rename TIME to destination_country
    df = df.rename(columns={"TIME": "destination_country"})

    # Add the entry reason
    df["motif"] = motif

    # Reshape to long format
    df = df.melt(
        id_vars=["destination_country", "motif"],
        var_name="year",
        value_name="permits"
    )

    # Clean data types
    df["year"] = df["year"].astype(int)

    df["permits"] = (
        pd.to_numeric(df["permits"], errors="coerce")
        .fillna(0)
        .astype(int)
    )

    # Reorder columns
    df = df[
        ["destination_country", "motif", "year", "permits"]
    ]

    return df


In [ ]:
sheet3_raw = clean_and_reshape_first_permits(
    EUROPE_PATH / "eurostat_first_permits_cameroon.xlsx",
    sheet_name="Feuille 3",
    motif="travail"
)

sheet3_raw


In [ ]:
sheet4_raw = clean_and_reshape_first_permits(
    EUROPE_PATH / "eurostat_first_permits_cameroon.xlsx",
    sheet_name="Feuille 4",
    motif="autre"
)

sheet4_raw


In [ ]:
eurostat_first_permits_cleaned = pd.concat(
    [sheet1_raw, sheet2_raw, sheet3_raw, sheet4_raw],
    ignore_index=True
)

eurostat_first_permits_cleaned


In [ ]:
eurostat_first_permits_cleaned["source"] = "eurostat"

eurostat_first_permits_cleaned

In [ ]:
eurostat_first_permits_cleaned.to_csv(
    CLEANED_PATH / "eurostat_first_permits_cleaned.csv",
    index=False
)


In [ ]:
eurostat_first_permits_cleaned = eurostat_first_permits_cleaned.rename(
    columns={"motif": "entry_reasons"}
)

eurostat_first_permits_cleaned


In [ ]:
eurostat_first_permits_cleaned.to_csv(
    CLEANED_PATH / "eurostat_first_permits_cleaned.csv",
    index=False
)